In [6]:
# ============================================================
# Imports
# ============================================================

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap, RunnableLambda
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# ============================================================
# Hybrid Retriever (Dense + Sparse) setup with documents
# ============================================================

# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

### Sparse Retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ##top- k documents to retriever

## step 4 : Combine with Ensemble Retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weight=[0.7,0.3]
)
# ============================================================
# Prompt
# ============================================================

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the user's question ONLY using the provided context.

If the answer is not available in the context, say:
"I don't have enough information."

--------------------
Context:
{context}
--------------------

Question:
{question}
""")

# ============================================================
# LLM
# ============================================================

llm = init_chat_model(
    "openai:gpt-3.5-turbo",
    temperature=0.2
)

# ============================================================
# Helper Functions
# ============================================================

def retrieve_documents(inputs):
    """
    Retrieve relevant documents using Hybrid Retriever.
    """
    return hybrid_retriever.invoke(inputs["question"])


def format_documents(docs):
    """
    Convert Documents -> Single Context String
    """
    return "\n\n".join(doc.page_content for doc in docs)


# ============================================================
# Retrieval Pipeline
# ============================================================

retrieval_chain = RunnableMap(
    {
        "question": lambda x: x["question"],
        "docs": RunnableLambda(retrieve_documents),
    }
)

# ============================================================
# Context Builder
# ============================================================

context_chain = RunnableMap(
    {
        "question": lambda x: x["question"],
        "docs": lambda x: x["docs"],
        "context": lambda x: format_documents(x["docs"]),
    }
)

# ============================================================
# Answer Generator
# ============================================================

answer_chain = RunnableMap(
    {
        "answer": (
            RunnableMap(
                {
                    "context": lambda x: x["context"],
                    "question": lambda x: x["question"],
                }
            )
            | prompt
            | llm
            | StrOutputParser()
        ),
        "docs": lambda x: x["docs"],
        "context": lambda x: x["context"],
        "question": lambda x: x["question"],
    }
)

# ============================================================
# Final LCEL RAG Chain
# ============================================================

rag_chain = (
    retrieval_chain
    | context_chain
    | answer_chain
)

# ============================================================
# Ask Question
# ============================================================

result = rag_chain.invoke(
    {
        "question": "How can I build an app using LLMs?"
    }
)

# ============================================================
# Output
# ============================================================
print("=" * 80)
print("🤖 QUESTION")
print("=" * 80)
print(result["question"])

print("\n")

print("=" * 80)
print("✅ ANSWER")
print("=" * 80)
print(result["answer"])

print("\n")

print("=" * 80)
print(f"📚 RETRIEVED DOCUMENTS ({len(result['docs'])})")
print("=" * 80)

for i, doc in enumerate(result["docs"], start=1):
    print(f"\nDocument {i}")
    print("-" * 80)
    print(doc.page_content)

    if doc.metadata:
        print("\nMetadata:")
        for key, value in doc.metadata.items():
            print(f"  {key}: {value}")

print("\n")
# print("\n")

# print("=" * 80)
# print("SOURCE DOCUMENTS")
# print("=" * 80)

# for i, doc in enumerate(result["docs"], start=1):
#     print(f"\nDocument {i}")
#     print("-" * 80)
#     print(doc.page_content)

# print("\n")

# print("=" * 80)
# print("CONTEXT SENT TO LLM")
# print("=" * 80)
# print(result["context"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2845.58it/s]


🤖 QUESTION
How can I build an app using LLMs?


✅ ANSWER
LangChain helps build LLM applications.


📚 RETRIEVED DOCUMENTS (4)

Document 1
--------------------------------------------------------------------------------
LangChain helps build LLM applications.

Document 2
--------------------------------------------------------------------------------
Langchain can be used to develop agentic ai application.

Document 3
--------------------------------------------------------------------------------
Langchain has many types of retrievers.

Document 4
--------------------------------------------------------------------------------
Pinecone is a vector database for semantic search.


